# **Correlacions de Variables Meteorològiques**

Anàlisi de correlacions entre variables meteorològiques a les estacions de **Sabadell Nord** (Parc Agrari) i **Vacarisses**, amb dades cada 30 minuts.

**Variables analitzades:** Temperatura, Humitat, Precipitació, Pressió i Vent.

**Mètodes:** Correlació de Pearson (lineal) i Spearman (monòtona).

## 1. Càrrega i Preparació de Dades

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

BASE = 'https://raw.githubusercontent.com/DavidDDRC99/VO-evolution/refs/heads/main/Cleaned%20Data/'

df_sbd = pd.read_csv(BASE + 'Sbd_nord_hourly.csv')
df_vac = pd.read_csv(BASE + 'Vacarisses_hourly.csv')

df_sbd['datetime_utc'] = pd.to_datetime(df_sbd['datetime_utc'])
df_vac['datetime_utc'] = pd.to_datetime(df_vac['datetime_utc'])

df_sbd.rename(columns={
    'humidity (%)': 'humidity',
    'rain_mm': 'rain',
    'pressure (hPa)': 'pressure',
    'radiation (W/m²)': 'radiation',
    'avg_wind_kmh': 'wind_avg',
    'max_wind_kmh': 'wind_max'
}, inplace=True)

df_vac.rename(columns={
    'humidity (%)': 'humidity',
    'rain_mm': 'rain',
    'pressure (hPa)': 'pressure',
    'radiation (W/m²)': 'radiation'
}, inplace=True)

print(f"Sabadell Nord: {len(df_sbd)} registres | {df_sbd['datetime_utc'].min()} → {df_sbd['datetime_utc'].max()}")
print(f"Vacarisses:    {len(df_vac)} registres | {df_vac['datetime_utc'].min()} → {df_vac['datetime_utc'].max()}")

In [1]:
def assign_season(month):
    if month in [12, 1, 2]:
        return 'Hivern'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Estiu'
    else:
        return 'Tardor'

def assign_time_slot(hour):
    if 6 <= hour < 12:
        return 'Matí'
    elif 12 <= hour < 18:
        return 'Tarda'
    elif 18 <= hour < 22:
        return 'Vespre'
    else:
        return 'Nit'

for df in [df_sbd, df_vac]:
    df['season'] = df['datetime_utc'].dt.month.map(assign_season)
    df['time_slot'] = df['datetime_utc'].dt.hour.map(assign_time_slot)

season_order = ['Primavera', 'Estiu', 'Tardor', 'Hivern']
time_order = ['Matí', 'Tarda', 'Vespre', 'Nit']

palette_season = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']
palette_time = ['#f1c40f', '#e67e22', '#9b59b6', '#34495e']

print("Columnes Sbd Nord:", list(df_sbd.columns))
print("Columnes Vacarisses:", list(df_vac.columns))

## 2. Matrius de Correlació

### 2.1 Sabadell Nord

In [1]:
def plot_corr_matrix(df, station_name):
    vars_corr = ['T_avg', 'T_max', 'T_min', 'humidity', 'rain', 'pressure']
    if 'wind_avg' in df.columns:
        vars_corr.extend(['wind_avg', 'wind_max'])

    df_corr = df[vars_corr].dropna()
    pearson = df_corr.corr(method='pearson')
    spearman = df_corr.corr(method='spearman')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(pearson, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=axes[0], cbar_kws={'label': 'Pearson'})
    axes[0].set_title(f'Pearson — {station_name}', fontsize=13, fontweight='bold')
    sns.heatmap(spearman, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=axes[1], cbar_kws={'label': 'Spearman'})
    axes[1].set_title(f'Spearman — {station_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_corr_matrix(df_sbd, 'Sabadell Nord')

### 2.2 Vacarisses

In [1]:
plot_corr_matrix(df_vac, 'Vacarisses')

## 3. Scatter Plots

Cada parella de variables es mostra amb dos scatter plots:
- **Per estació** (Primavera, Estiu, Tardor, Hivern)
- **Per franja horària** (Matí, Tarda, Vespre, Nit)

S'usa transparència (alpha=0.3) per gestionar la densitat de punts.

### 3.1 Temperatura vs Humitat

In [1]:
def scatter_T_avg_humidity(df, station, hue_col, hue_order, palette):
    data = df[['T_avg', 'humidity', 'season', 'time_slot']].dropna().copy()
    r_pearson, _ = stats.pearsonr(data['T_avg'], data['humidity'])
    r_spearman, _ = stats.spearmanr(data['T_avg'], data['humidity'])

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, cat in enumerate(hue_order):
        subset = data[data[hue_col] == cat]
        ax.scatter(subset['T_avg'], subset['humidity'],
                   alpha=0.3, s=4, label=cat, color=palette[i])
    ax.set_xlabel('Temperatura mitjana (°C)')
    ax.set_ylabel('Humitat relativa (%)')
    ax.set_title(f'{station} — r_Pearson={r_pearson:.3f}, r_Spearman={r_spearman:.3f}')
    ax.legend(markerscale=4)
    plt.tight_layout()
    plt.show()

scatter_T_avg_humidity(df_sbd, 'Sabadell Nord', 'season', season_order, palette_season)
scatter_T_avg_humidity(df_vac, 'Vacarisses', 'season', season_order, palette_season)

scatter_T_avg_humidity(df_sbd, 'Sabadell Nord', 'time_slot', time_order, palette_time)
scatter_T_avg_humidity(df_vac, 'Vacarisses', 'time_slot', time_order, palette_time)


### 3.2 Temperatura vs Pluja (>0.2 mm)

In [1]:
def scatter_T_avg_rain(df, station, hue_col, hue_order, palette):
    data = df[['T_avg', 'rain', 'season', 'time_slot']].dropna().copy()
    data = data[data['rain'] > 0.2].copy()
    r_pearson, _ = stats.pearsonr(data['T_avg'], data['rain'])
    r_spearman, _ = stats.spearmanr(data['T_avg'], data['rain'])

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, cat in enumerate(hue_order):
        subset = data[data[hue_col] == cat]
        ax.scatter(subset['T_avg'], subset['rain'],
                   alpha=0.3, s=4, label=cat, color=palette[i])
    ax.set_xlabel('Temperatura mitjana (°C)')
    ax.set_ylabel('Precipitació (mm)')
    ax.set_title(f'{station} — r_Pearson={r_pearson:.3f}, r_Spearman={r_spearman:.3f}')
    ax.legend(markerscale=4)
    plt.tight_layout()
    plt.show()

scatter_T_avg_rain(df_sbd, 'Sabadell Nord', 'season', season_order, palette_season)
scatter_T_avg_rain(df_vac, 'Vacarisses', 'season', season_order, palette_season)

scatter_T_avg_rain(df_sbd, 'Sabadell Nord', 'time_slot', time_order, palette_time)
scatter_T_avg_rain(df_vac, 'Vacarisses', 'time_slot', time_order, palette_time)


### 3.3 Pressió vs Pluja (>0.2 mm)

In [1]:
def scatter_pressure_rain(df, station, hue_col, hue_order, palette):
    data = df[['pressure', 'rain', 'season', 'time_slot']].dropna().copy()
    data = data[data['rain'] > 0.2].copy()
    r_pearson, _ = stats.pearsonr(data['pressure'], data['rain'])
    r_spearman, _ = stats.spearmanr(data['pressure'], data['rain'])

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, cat in enumerate(hue_order):
        subset = data[data[hue_col] == cat]
        ax.scatter(subset['pressure'], subset['rain'],
                   alpha=0.3, s=4, label=cat, color=palette[i])
    ax.set_xlabel('Pressió (hPa)')
    ax.set_ylabel('Precipitació (mm)')
    ax.set_title(f'{station} — r_Pearson={r_pearson:.3f}, r_Spearman={r_spearman:.3f}')
    ax.legend(markerscale=4)
    plt.tight_layout()
    plt.show()

scatter_pressure_rain(df_sbd, 'Sabadell Nord', 'season', season_order, palette_season)
scatter_pressure_rain(df_vac, 'Vacarisses', 'season', season_order, palette_season)

scatter_pressure_rain(df_sbd, 'Sabadell Nord', 'time_slot', time_order, palette_time)
scatter_pressure_rain(df_vac, 'Vacarisses', 'time_slot', time_order, palette_time)


### 3.4 Pressió vs Temperatura

In [1]:
def scatter_pressure_T_avg(df, station, hue_col, hue_order, palette):
    data = df[['pressure', 'T_avg', 'season', 'time_slot']].dropna().copy()
    r_pearson, _ = stats.pearsonr(data['pressure'], data['T_avg'])
    r_spearman, _ = stats.spearmanr(data['pressure'], data['T_avg'])

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, cat in enumerate(hue_order):
        subset = data[data[hue_col] == cat]
        ax.scatter(subset['pressure'], subset['T_avg'],
                   alpha=0.3, s=4, label=cat, color=palette[i])
    ax.set_xlabel('Pressió (hPa)')
    ax.set_ylabel('Temperatura mitjana (°C)')
    ax.set_title(f'{station} — r_Pearson={r_pearson:.3f}, r_Spearman={r_spearman:.3f}')
    ax.legend(markerscale=4)
    plt.tight_layout()
    plt.show()

scatter_pressure_T_avg(df_sbd, 'Sabadell Nord', 'season', season_order, palette_season)
scatter_pressure_T_avg(df_vac, 'Vacarisses', 'season', season_order, palette_season)

scatter_pressure_T_avg(df_sbd, 'Sabadell Nord', 'time_slot', time_order, palette_time)
scatter_pressure_T_avg(df_vac, 'Vacarisses', 'time_slot', time_order, palette_time)


### 3.5 Velocitat del Vent vs Pluja (>0.2 mm)

In [1]:
def scatter_wind_avg_rain(df, station, hue_col, hue_order, palette):
    data = df[['wind_avg', 'rain', 'season', 'time_slot']].dropna().copy()
    data = data[data['rain'] > 0.2].copy()
    r_pearson, _ = stats.pearsonr(data['wind_avg'], data['rain'])
    r_spearman, _ = stats.spearmanr(data['wind_avg'], data['rain'])

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, cat in enumerate(hue_order):
        subset = data[data[hue_col] == cat]
        ax.scatter(subset['wind_avg'], subset['rain'],
                   alpha=0.3, s=4, label=cat, color=palette[i])
    ax.set_xlabel('Velocitat del vent mitjana (km/h)')
    ax.set_ylabel('Precipitació (mm)')
    ax.set_title(f'{station} — r_Pearson={r_pearson:.3f}, r_Spearman={r_spearman:.3f}')
    ax.legend(markerscale=4)
    plt.tight_layout()
    plt.show()

scatter_wind_avg_rain(df_sbd, 'Sabadell Nord', 'season', season_order, palette_season)

scatter_wind_avg_rain(df_sbd, 'Sabadell Nord', 'time_slot', time_order, palette_time)
